# Chapter 6 — Safe Tool Execution and Tool Testing

*The execution point is the danger point. Three gates, contract tests, no exceptions.*

## Objective

**Why.** A tool call is where the agent acts on the world — the step the audit log records and a regulator later asks about — so it deserves a checkpoint, not a bare `fn(**args)` call.

**What.** Between deciding to act and acting, the agent runs independent checks, each able to refuse the call and record why (defense in depth). Three classes of failure, in a deliberate order: **well-formed** (syntax) → **permitted** (policy) → **plausible** (plausibility).

**How.** We wire a `ToolRegistry` into a `GovernedToolExecutor`, watch each gate fire, then fuzz a tool to show that tool implementations themselves need testing.

In [ ]:
from pydantic import BaseModel
from glassloop.core import ToolCall
from glassloop.tools import (
    GovernedToolExecutor, PlausibilityGate, PolicyGate, RiskLevel,
    SyntaxGate, Tool, ToolRegistry,
)
from glassloop.tools.executor import GateDecision, GateResult
from glassloop.tools.testing import fuzz_tool

## A simple tool

In [ ]:
class AdderIn(BaseModel):
    x: int
class AdderOut(BaseModel):
    y: int

def _add_one(x: int) -> dict:
    return {'y': x + 1}

adder = Tool(name='adder', description='add one to x',
             input_schema=AdderIn, output_schema=AdderOut,
             risk=RiskLevel.LOW, fn=_add_one)

registry = ToolRegistry()
registry.register(adder)

## The three gates

The checkpoint, made concrete. `GovernedToolExecutor` runs the three checks in order; a refusal at any layer stops the call before it runs, and every verdict is part of the audit record.

```
syntax → policy → plausibility → tool fn
```

In [ ]:
executor = GovernedToolExecutor(registry)  # uses the three default gates

ok = executor.execute(ToolCall(tool_name='adder', arguments={'x': 5}))
print('success:', ok.success, 'output:', ok.output)

bad = executor.execute(ToolCall(tool_name='adder', arguments={'x': 'not-an-int'}))
print('success:', bad.success)
print('error:  ', bad.error)
print('gates: ', [(g.gate_name, g.decision.value) for g in bad.gate_results])

## Policy as code

Domain rules aren't universal — they depend on the business, the jurisdiction, the moment. *Policy-as-code* writes each rule as a small, readable function, so a reviewer can read the rule and the audit log together and see why a call was allowed or denied. Here a policy is a `(action, state) -> GateResult` callable; Chapter 12 builds a full banking library on this surface.

In [ ]:
def small_x_only(action, state):
    if action.tool_name == 'adder' and action.arguments.get('x', 0) > 100:
        return GateResult(GateDecision.DENY, 'small_x_only', 'x must be <= 100')
    return GateResult(GateDecision.ALLOW, 'small_x_only')

policy_exec = GovernedToolExecutor(registry,
    gates=[SyntaxGate(), PolicyGate(policies=[small_x_only]), PlausibilityGate()])

blocked = policy_exec.execute(ToolCall(tool_name='adder', arguments={'x': 1000}))
print('blocked:', blocked.error)

## Plausibility: refusing the outlier

Once a call is well-formed and permitted, a long tail of valid-but-abnormal calls remains — and abnormal is where attacks and bugs live. A plausibility check refuses outliers without needing to understand them, by encoding a cheap expectation of what a normal call looks like. The default expectation is a size bound (it catches prompt-injection payloads that smuggle large strings into tool args).

In [ ]:
tight = GovernedToolExecutor(registry, gates=[SyntaxGate(), PlausibilityGate(max_args_size=50)])
result = tight.execute(ToolCall(tool_name='adder', arguments={'x': int('1' * 60)}))
print('blocked:', result.error)

## An optional upgrade: a geometric plausibility gate

The default `PlausibilityGate` only checks argument size — it can't tell that calling `draft_response` from the `classify` state skips three workflow steps. This is the book's **first use of GMS**: a trained store, loaded from disk, that scores each `(state, has_enables, next_step)` triple geometrically. `GMSPlausibilityGate` denies calls above a calibrated threshold θ (read from `calibration.json`). **Optional** — reach for it when the workflow is safety-critical; the size check suffices otherwise.

In [ ]:
import json, torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from glassloop.gms_backend import GMSPlausibilityGate

_root = Path('.') if Path('data/gms_banking_store').exists() else Path('..')
store = GMSExpertStore(
    DocGMSConfig(store_path=str(_root / 'data' / 'gms_banking_store')),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)
store.load()
theta = json.loads(
    (_root / 'data' / 'gms_banking_store' / 'calibration.json').read_text()
)['plausibility_gate']['threshold']                       # 1.35, calibrated in App. C

gms_gate = GMSPlausibilityGate(store, theta=theta,
                               context='classify', relation='has_enables')
for tool in ('extract', 'search_policy', 'draft_response', 'wire_international'):
    score = store.score_triple('classify', 'has_enables', tool)
    verdict = gms_gate.check(ToolCall(tool_name=tool, arguments={}), None, registry)
    print(f'  classify -> {tool:<18} score={score:.3f}  {verdict.decision.value}')

## Tool contract testing

A well-behaved tool either accepts an input or raises `ValidationError`. Anything else means an uncovered input path that will crash in production. `fuzz_tool` reports the three counters.

In [ ]:
report = fuzz_tool(adder, num_cases=200, seed=1)
print(f'accepted:         {report["accepted"]}')
print(f'rejected cleanly: {report["rejected_clean"]}')
print(f'crashes:          {report["crashed"]}')

## Anti-patterns flagged here

- Catching all exceptions and continuing.
- Retrying a write-side-effect tool without idempotency.
- Trusting a tool's output schema because *the docs say so*.

In [ ]:
# Self-check
assert ok.success and ok.output == {'y': 6}
assert not bad.success
assert not blocked.success and 'small_x_only' in blocked.error
assert report['crashed'] == 0
print('OK')

## Exercises

Worked solutions follow. They extend the executor and its gates toward the banking domain the capstone uses. Exercise 6.3 reuses the GMS store loaded above.

In [ ]:
# Exercise 6.1 — a policy gate for reversal authority
class ReverseIn(BaseModel):
    amount: float
class ReverseOut(BaseModel):
    ok: bool

reverse_fee = Tool(name='reverse_fee', description='reverse a fee',
                   input_schema=ReverseIn, output_schema=ReverseOut,
                   risk=RiskLevel.MEDIUM, fn=lambda amount: {'ok': True})
reg = ToolRegistry(); reg.register(reverse_fee)

REP_LIMIT = 35.0
def within_rep_authority(action, state):
    if action.tool_name == 'reverse_fee' and float(action.arguments.get('amount', 0)) > REP_LIMIT:
        return GateResult(GateDecision.ESCALATE, 'within_rep_authority',
                          f'amount > {REP_LIMIT} needs supervisor')
    return GateResult(GateDecision.ALLOW, 'within_rep_authority')

ex = GovernedToolExecutor(reg, gates=[SyntaxGate(), PolicyGate(policies=[within_rep_authority]), PlausibilityGate()])
print('reverse $30 :', ex.execute(ToolCall(tool_name='reverse_fee', arguments={'amount': 30})).success)
r = ex.execute(ToolCall(tool_name='reverse_fee', arguments={'amount': 100}))
print('reverse $100:', r.success, '|', [(g.gate_name, g.decision.value) for g in r.gate_results])

In [ ]:
# Exercise 6.2 — make fuzz_tool find a crash, then fix the contract
from pydantic import field_validator

class CodeIn(BaseModel):
    code: str = 'n/a'            # default is non-numeric
class CodeOut(BaseModel):
    n: int
buggy = Tool(name='parse_code', description='parse a numeric code',
             input_schema=CodeIn, output_schema=CodeOut,
             risk=RiskLevel.LOW, fn=lambda code: {'n': int(code)})
print('buggy crashes:', fuzz_tool(buggy, num_cases=200, seed=1)['crashed'])

class CodeInFixed(BaseModel):
    code: str = '0'
    @field_validator('code')
    @classmethod
    def numeric(cls, v):
        if not str(v).lstrip('-').isdigit():
            raise ValueError('code must be numeric')
        return v
fixed = Tool(name='parse_code2', description='parse a numeric code',
             input_schema=CodeInFixed, output_schema=CodeOut,
             risk=RiskLevel.LOW, fn=lambda code: {'n': int(code)})
print('fixed crashes:', fuzz_tool(fixed, num_cases=200, seed=1)['crashed'])

In [ ]:
# Exercise 6.3 — calibrating the plausibility threshold
for theta_try in (1.0, 1.35, 1.5):
    verdicts = {
        t: ('allow' if store.score_triple('classify', 'has_enables', t) <= theta_try else 'deny')
        for t in ('extract', 'search_policy', 'draft_response', 'wire_international')
    }
    print(f'theta={theta_try}: {verdicts}')
# theta=1.0 denies the legitimate `extract` too (false-deny); theta=1.5 admits
# `search_policy` (false-allow — an illegal next-step let through). A bank bounds
# false-allows above all, so 1.35 is the operating point: 0% false-allow at the
# cost of one borderline false-deny.